# Capa Gold — Preguntas de negocio

**Qué hace esta notebook:** crea el schema `dev.gold_ads` y una tabla por pregunta de negocio, combinando las tablas silver con joins, agregaciones, funciones de ventana y columnas derivadas.

| # | Pregunta | Tabla gold |
|---|---|---|
| 1 | ¿Qué industria concentra más presupuesto y en qué canales? | `q1_presupuesto_por_industria_canal` |
| 2 | ¿Qué % de su presupuesto ya gastó cada campaña? | `q2_presupuesto_usado_por_campana` |
| 3 | ¿Qué canal tiene mejor CTR? | `q3_ctr_por_canal` |
| 4 | ¿Cuál es el costo por conversión (CPA) por canal? | `q4_cpa_por_canal` |
| 5 | ¿En qué etapa del embudo se pierde más gente por canal? | `q5_embudo_por_canal` |
| — | *(tabla puente)* atribución de clics y conversiones, escrita una sola vez | `clicks_atribuidos` + `conversions_atribuidas` |

**La regla de atribución (la pieza clave del gold):** en silver los eventos quedaron válidos, pero el `id` de un clic se repite entre archivos: un clic con `impression_id = 7` puede corresponder a la impresión 7 de octubre o a la de noviembre. Para pasar del clic a su campaña/canal hay que elegir UNA impresión. Regla usada (estándar en AdTech):

> El clic pertenece a la impresión con ese id **más cercana en el tiempo**, **prefiriendo la anterior al clic** (un clic real ocurre DESPUÉS de su impresión).

Se implementa con `row_number() OVER (PARTITION BY el_clic ORDER BY impresion_anterior_primero, cercania_en_segundos) = 1` — una función de ventana que des-ambigua. Mismo criterio para conversiones → clics. **La regla vive UNA sola vez en las tablas puente** `clicks_atribuidos` y `conversions_atribuidas`; las preguntas solo joinean contra ellas, así todas quedan consistentes entre sí.

**Idempotencia en gold:** cada tabla se recalcula completa desde silver con `CREATE OR REPLACE TABLE AS`. Re-ejecutar N veces produce exactamente lo mismo (el resultado no depende de corridas anteriores). No hace falta MERGE acá porque una tabla agregada se puede recalcular barato (los eventos son ~160 mil filas); el MERGE incremental ya vive en la capa silver.

In [0]:
%sql
-- Paso 0: schema de la capa gold (mismo patrón idempotente que silver).
CREATE SCHEMA IF NOT EXISTS dev.gold_ads
COMMENT 'Capa gold del dominio ads: una tabla agregada por pregunta de negocio';

In [0]:
%sql
-- PREGUNTA 1: ¿Qué industria concentra más presupuesto diario y en qué canales lo pone?
-- Join necesario: campaigns -> advertisers (industry) vía advertiser_id.
-- Agregaciones: COUNT campañas, SUM y AVG del presupuesto diario.
-- Función de ventana: SUM() OVER () para calcular el % del total 
-- Nulos: las 31 campañas sin presupuesto (era $0) no suman al total; 

CREATE OR REPLACE TABLE dev.gold_ads.q1_presupuesto_por_industria_canal AS
SELECT
  a.industry                                        AS industria,
  c.channel                                         AS canal,
  COUNT(*)                                          AS campanas,
  SUM(CASE WHEN c.daily_budget_usd IS NULL THEN 1 ELSE 0 END) AS campanas_sin_presupuesto,
  ROUND(SUM(c.daily_budget_usd), 2)                 AS presupuesto_diario_total_usd,
  ROUND(AVG(c.daily_budget_usd), 2)                AS presupuesto_diario_promedio_usd,
  ROUND(100.0 * SUM(c.daily_budget_usd)
        / SUM(SUM(c.daily_budget_usd)) OVER (), 2)  AS pct_del_presupuesto_total
FROM dev.silver_ads.campaigns c
JOIN dev.silver_ads.advertisers a ON a.id = c.advertiser_id
GROUP BY a.industry, c.channel;

-- Vista rápida de la respuesta: las 5 combinaciones industria-canal con más presupuesto
SELECT industria, canal, campanas, presupuesto_diario_total_usd, pct_del_presupuesto_total
FROM dev.gold_ads.q1_presupuesto_por_industria_canal
ORDER BY presupuesto_diario_total_usd DESC
LIMIT 5;

industria,canal,campanas,presupuesto_diario_total_usd,pct_del_presupuesto_total
Gaming,Social,141,220407.6,7.27
Gaming,Display,117,187941.13,6.19
FinTech,Search,127,184976.13,6.1
FinTech,Display,121,177273.06,5.84
Gaming,Video,117,174992.86,5.77


In [0]:
%sql
-- TABLA PUENTE: ATRIBUCIÓN de clicks a conversiones.

--  ante varias impresiones con el mismo id, el clic pertenece a la más cercana en el tiempo, prefiriendo la ANTERIOR al clic (un clic real ocurre después de
-- su impresión). Mismo criterio para conversión -> clic. Se implementa con la función de
-- ventana row_number() y queda fija en 2 tablas:
--   clicks_atribuidos       (1 fila por clic)        -> campaña y canal de cada clic
--   conversions_atribuidas (1 fila por conversión)  -> campaña y canal de cada conversión
-- Idempotencia: CREATE OR REPLACE (recalcula todo desde silver en cada corrida).

CREATE OR REPLACE TABLE dev.gold_ads.clicks_atribuidos AS
SELECT
  k.event_key AS click_key,
  k.cost_usd  AS click_cost_usd,
  a.campaign_id,
  c.channel   AS canal,
  i.country   AS pais
FROM dev.silver_ads.clicks k
JOIN dev.silver_ads.impressions i ON i.id = k.impression_id
JOIN dev.silver_ads.ads a ON a.id = i.ad_id
JOIN dev.silver_ads.campaigns c ON c.id = a.campaign_id
QUALIFY row_number() OVER (
  PARTITION BY k.event_key
  ORDER BY (i.event_ts > k.event_ts) ASC,                      -- primero la impresión previa al clic
           abs(timestampdiff(SECOND, i.event_ts, k.event_ts))  -- y la más cercana en el tiempo
) = 1;

CREATE OR REPLACE TABLE dev.gold_ads.conversions_atribuidas AS
WITH conv_a_click AS (
  SELECT
    v.event_key AS conv_key,
    v.value_usd,
    v.event_ts  AS conv_ts,
    k.event_key AS click_key
  FROM dev.silver_ads.conversions v
  JOIN dev.silver_ads.clicks k ON k.id = v.click_id
  QUALIFY row_number() OVER (
    PARTITION BY v.event_key
    ORDER BY (k.event_ts > v.event_ts) ASC,
             abs(timestampdiff(SECOND, k.event_ts, v.event_ts))
  ) = 1
)
SELECT ca.conv_key, ca.value_usd, ca.conv_ts, ca.click_key, at.campaign_id, at.canal, at.pais
FROM conv_a_click ca
JOIN dev.gold_ads.clicks_atribuidos at ON at.click_key = ca.click_key;

-- Chequeo rápido: tienen que ser EXACTAMENTE todos los clics y todas las conversiones
-- (la atribución elige padre, nunca pierde ni duplica eventos).
SELECT
  (SELECT COUNT(*) FROM dev.gold_ads.clicks_atribuidos)       AS clics_atribuidos,
  (SELECT COUNT(*) FROM dev.silver_ads.clicks)                 AS clics_en_silver,
  (SELECT COUNT(*) FROM dev.gold_ads.conversions_atribuidas)   AS conv_atribuidas,
  (SELECT COUNT(*) FROM dev.silver_ads.conversions)            AS conv_en_silver;

clics_atribuidos,clics_en_silver,conv_atribuidas,conv_en_silver
3184,3184,256,256


In [0]:
%sql
-- PREGUNTA 2: ¿Qué % de su presupuesto diario ya quemó cada campaña?
-- Joins: impressions -> ads -> campaigns (gasto de impresiones)
--         clicks_atribuidos (gasto de clics: la atribución ya está resuelta en la tabla puente)
-- Columnas derivadas: presupuesto_periodo = diario x días activos; pct_gastado.
-- Función de ventana: RANK() para rankear las campañas más gastadoras.

CREATE OR REPLACE TABLE dev.gold_ads.q2_presupuesto_usado_por_campana AS
WITH gasto_imp AS (
  SELECT a.campaign_id,
         COUNT(DISTINCT i.event_key)         AS impresiones,
         COUNT(DISTINCT to_date(i.event_ts)) AS dias_activos,
         COALESCE(SUM(i.cost_usd), 0)        AS gasto_impresiones_usd
  FROM dev.silver_ads.impressions i
  JOIN dev.silver_ads.ads a ON a.id = i.ad_id
  GROUP BY a.campaign_id
),
gasto_clicks AS (
  SELECT campaign_id, COALESCE(SUM(click_cost_usd), 0) AS gasto_clics_usd
  FROM dev.gold_ads.clicks_atribuidos
  GROUP BY campaign_id
)
SELECT
  c.id                        AS campaign_id,
  c.campaign_name,
  c.channel                   AS canal,
  c.daily_budget_usd          AS presupuesto_diario_usd,
  COALESCE(gi.impresiones, 0) AS impresiones,
  gi.dias_activos,
  ROUND(COALESCE(gi.gasto_impresiones_usd, 0) + COALESCE(gc.gasto_clics_usd, 0), 4) AS gasto_real_usd,
  ROUND(c.daily_budget_usd * gi.dias_activos, 2) AS presupuesto_periodo_usd,
  ROUND(100.0 * (COALESCE(gi.gasto_impresiones_usd, 0) + COALESCE(gc.gasto_clics_usd, 0))
        / (c.daily_budget_usd * gi.dias_activos), 2) AS pct_gastado
FROM dev.silver_ads.campaigns c
LEFT JOIN gasto_imp gi    ON gi.campaign_id = c.id
LEFT JOIN gasto_clicks gc ON gc.campaign_id = c.id;

-- Vista rápida: top 5 campañas por % de presupuesto quemado (función de ventana RANK)
SELECT campaign_id, campaign_name, canal, pct_gastado,
       RANK() OVER (ORDER BY pct_gastado DESC NULLS LAST) AS ranking_gastadoras
FROM dev.gold_ads.q2_presupuesto_usado_por_campana
ORDER BY pct_gastado DESC NULLS LAST
LIMIT 5;

campaign_id,campaign_name,canal,pct_gastado,ranking_gastadoras
699,Revolutionize Collaborative Bandwidth,Display,0.79,1
312,Synergize Real-Time Applications,Search,0.35,2
337,Extend Plug-And-Play E-Markets,Display,0.3,3
1677,Exploit Front-End Metrics,Display,0.28,4
138,Orchestrate Cross-Media Info-Mediaries,Video,0.26,5


In [0]:
%sql
-- PREGUNTA 3: ¿Qué canal tiene mejor CTR (clics / impresiones)?
-- Joins: impresiones -> ads -> campaigns (canal); clicks_atribuidos (canal ya atribuido).
-- Métrica derivada: ctr_pct = 100 * clics / impresiones.
-- Función de ventana: DENSE_RANK() para rankear canales por CTR.

CREATE OR REPLACE TABLE dev.gold_ads.q3_ctr_por_canal AS
WITH imp AS (
  SELECT c.channel AS canal, COUNT(*) AS impresiones
  FROM dev.silver_ads.impressions i
  JOIN dev.silver_ads.ads a ON a.id = i.ad_id
  JOIN dev.silver_ads.campaigns c ON c.id = a.campaign_id
  GROUP BY c.channel
),
clk AS (
  SELECT canal, COUNT(*) AS clics
  FROM dev.gold_ads.clicks_atribuidos
  GROUP BY canal
)
SELECT
  imp.canal,
  imp.impresiones,
  COALESCE(clk.clics, 0) AS clics,
  ROUND(100.0 * COALESCE(clk.clics, 0) / imp.impresiones, 3) AS ctr_pct,
  DENSE_RANK() OVER (ORDER BY ROUND(100.0 * COALESCE(clk.clics, 0) / imp.impresiones, 3) DESC) AS ranking_ctr
FROM imp
LEFT JOIN clk ON clk.canal = imp.canal;

SELECT * FROM dev.gold_ads.q3_ctr_por_canal ORDER BY ranking_ctr;

canal,impresiones,clics,ctr_pct,ranking_ctr
Search,38718,820,2.118,1
Video,40081,785,1.959,2
Social,41015,799,1.948,3
Display,40186,780,1.941,4


In [0]:
%sql
-- PREGUNTA 4: ¿Cuánto cuesta cada conversión (CPA) por canal?
-- Joins: impresiones -> canal (gasto); clicks_atribuidos (gasto de clics por canal);
--        conversions_atribuidas (conversiones por canal: la doble atribución ya está resuelta).
-- Métrica derivada: cpa_usd = gasto_total / conversiones.

CREATE OR REPLACE TABLE dev.gold_ads.q4_cpa_por_canal AS
WITH gasto_imp AS (
  SELECT c.channel AS canal, COALESCE(SUM(i.cost_usd), 0) AS gasto_impresiones_usd
  FROM dev.silver_ads.impressions i
  JOIN dev.silver_ads.ads a ON a.id = i.ad_id
  JOIN dev.silver_ads.campaigns c ON c.id = a.campaign_id
  GROUP BY c.channel
),
gasto_clicks AS (
  SELECT canal, COALESCE(SUM(click_cost_usd), 0) AS gasto_clics_usd
  FROM dev.gold_ads.clicks_atribuidos
  GROUP BY canal
),
conv AS (
  SELECT canal, COUNT(*) AS conversiones, SUM(value_usd) AS ingresos_usd
  FROM dev.gold_ads.conversions_atribuidas
  GROUP BY canal
)
SELECT
  gi.canal,
  ROUND(gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0), 4) AS gasto_total_usd,
  COALESCE(cv.conversiones, 0) AS conversiones,
  ROUND((gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0))
        / NULLIF(cv.conversiones, 0), 4) AS cpa_usd,   -- NULL si un canal no convirtió (no divido por 0)
  ROUND(COALESCE(cv.ingresos_usd, 0), 2) AS ingresos_usd
FROM gasto_imp gi
LEFT JOIN gasto_clicks gc ON gc.canal = gi.canal
LEFT JOIN conv cv         ON cv.canal = gi.canal;

SELECT * FROM dev.gold_ads.q4_cpa_por_canal ORDER BY cpa_usd;

canal,gasto_total_usd,conversiones,cpa_usd,ingresos_usd
Display,1655.7313,76,21.7859,1745.66
Video,1623.6793,68,23.8776,1560.58
Social,1627.5276,63,25.8338,1353.71
Search,1669.2879,49,34.0671,1169.85


In [0]:
%sql
-- PREGUNTA 5: ¿En qué etapa del embudo se pierde más gente por canal?
-- Joins: las 3 fuentes ya vienen por canal: impresiones (join directo),
--        clicks_atribuidos y conversions_atribuidas (tablas puente).
-- Columna derivada clave: etapa_mayor_perdida = la etapa con la tasa MÁS BAJA
--   (a menor tasa, más gente se pierde en ese paso).

CREATE OR REPLACE TABLE dev.gold_ads.q5_embudo_por_canal AS
WITH imp AS (
  SELECT c.channel AS canal, COUNT(*) AS impresiones
  FROM dev.silver_ads.impressions i
  JOIN dev.silver_ads.ads a ON a.id = i.ad_id
  JOIN dev.silver_ads.campaigns c ON c.id = a.campaign_id
  GROUP BY c.channel
),
clk AS (SELECT canal, COUNT(*) AS clics FROM dev.gold_ads.clicks_atribuidos GROUP BY canal),
cnv AS (SELECT canal, COUNT(*) AS conversiones FROM dev.gold_ads.conversions_atribuidas GROUP BY canal)
SELECT
  imp.canal,
  imp.impresiones,
  COALESCE(clk.clics, 0) AS clics,
  COALESCE(cnv.conversiones, 0) AS conversiones,
  ROUND(100.0 * COALESCE(clk.clics, 0) / imp.impresiones, 3) AS tasa_impresion_a_clic_pct,
  ROUND(100.0 * COALESCE(cnv.conversiones, 0) / NULLIF(COALESCE(clk.clics, 0), 0), 3) AS tasa_clic_a_conversion_pct,
  CASE
    WHEN 100.0 * COALESCE(clk.clics, 0) / imp.impresiones
       < 100.0 * COALESCE(cnv.conversiones, 0) / NULLIF(COALESCE(clk.clics, 0), 0)
      THEN 'impresion -> clic'
    ELSE 'clic -> conversion'
  END AS etapa_mayor_perdida
FROM imp
LEFT JOIN clk ON clk.canal = imp.canal
LEFT JOIN cnv ON cnv.canal = imp.canal;

SELECT * FROM dev.gold_ads.q5_embudo_por_canal ORDER BY canal;

canal,impresiones,clics,conversiones,tasa_impresion_a_clic_pct,tasa_clic_a_conversion_pct,etapa_mayor_perdida
Display,40186,780,76,1.941,9.744,impresion -> clic
Search,38718,820,49,2.118,5.976,impresion -> clic
Social,41015,799,63,1.948,7.885,impresion -> clic
Video,40081,785,68,1.959,8.662,impresion -> clic


In [0]:
%sql
-- VERIFICACIÓN: consistencia del gold contra los totales de silver.
-- Todas las sumas por canal tienen que cerrar con los totales de la capa anterior:
--   impresiones = 160.000 | clics = 3.184 | conversiones = 256
--   CTR global ≈ 2% y CVR global ≈ 8% (los porcentajes con los que el generador fue diseñado).
-- Las tablas puente tienen que tener exactamente 1 fila por evento (sin perder ni duplicar):
--   si la atribución duplicara clics, los clics por canal sumarían de más.
SELECT
  (SELECT COUNT(*) FROM dev.gold_ads.clicks_atribuidos) AS clics_puente,
  (SELECT COUNT(*) FROM dev.silver_ads.clicks)          AS clics_silver,
  (SELECT COUNT(*) FROM dev.gold_ads.conversions_atribuidas) AS conv_puente,
  (SELECT COUNT(*) FROM dev.silver_ads.conversions)     AS conv_silver,
  (SELECT SUM(impresiones) FROM dev.gold_ads.q5_embudo_por_canal) AS impresiones_q5,
  (SELECT COUNT(*) FROM dev.silver_ads.impressions)      AS impresiones_silver,
  (SELECT SUM(clics) FROM dev.gold_ads.q5_embudo_por_canal) AS clics_q5,
  (SELECT SUM(conversiones) FROM dev.gold_ads.q5_embudo_por_canal) AS conversiones_q5,
  (SELECT ROUND(100.0 * SUM(clics) / SUM(impresiones), 3) FROM dev.gold_ads.q5_embudo_por_canal) AS ctr_global_pct,
  (SELECT ROUND(100.0 * SUM(conversiones) / SUM(clics), 3) FROM dev.gold_ads.q5_embudo_por_canal) AS cvr_global_pct,
  (SELECT SUM(campanas) FROM dev.gold_ads.q1_presupuesto_por_industria_canal) AS campanas_q1,
  (SELECT ROUND(SUM(pct_del_presupuesto_total), 1) FROM dev.gold_ads.q1_presupuesto_por_industria_canal) AS pct_total_q1;

clics_puente,clics_silver,conv_puente,conv_silver,impresiones_q5,impresiones_silver,clics_q5,conversiones_q5,ctr_global_pct,cvr_global_pct,campanas_q1,pct_total_q1
3184,3184,256,256,160000,160000,3184,256,1.990,8.040,2000,100.0


## Respuestas de negocio

**1. ¿Qué industria concentra más presupuesto y dónde?** **Gaming** . Su combinación más fuerte es Gaming-Social con $220.408 diarios (7,27% del total, arriba del 5% que tocaría si fuera pareja). FinTech es la segunda (Search $184.976, 6,1%).

**2. ¿Qué % del presupuesto ya quemó cada campaña?** Ninguna pasó del **0,79%** (campaña 699, Display). Lectura de negocio: los presupuestos parecen estar sobredimensionados

**3. ¿Qué canal tiene mejor CTR?** **Search (2,118%)**, seguido de Video (1,959%), Social (1,948%) y Display (1,941%). Diferencias chicas,el CTR es casi uniforme .

**4. ¿Cuánto cuesta cada conversión (CPA)?** **Display es el más eficiente ($21,79)** y Search el más caro ($34,07) pese a su mejor CTR: en Search los clics convierten peor (49 conversiones con 820 clics), así que cada conversión sale más caro.

**5. ¿Dónde se pierde más gente en el embudo?** En los 4 canales la pérdida mayor es **impresión → clic** (~2% de tasa) y no clic → conversión (6-10%): el problema parece ser capturar el clic, no lograr la conversion.

**Validación del pipeline:** CTR global 1,99% y CVR global 8,04%, exactamente los valores con los que el generador fue diseñado (2% y 8%), y los totales por canal cierran contra silver (160.000 impresiones, 3.184 clics, 256 conversiones). Todo el embudo bronze → silver → gold es coherente de punta a punta.

In [0]:
%sql
-- GRÁFICO 1: CPA por canal (barras, de menor a mayor costo).
-- Aporta valor porque la historia es una COMPARACIÓN: Display $21,79 vs Search $34,07
-- en una sola imagen, sin leer una tabla.
SELECT canal, cpa_usd
FROM dev.gold_ads.q4_cpa_por_canal
ORDER BY cpa_usd;

canal,cpa_usd
Display,21.7859
Video,23.8776
Social,25.8338
Search,34.0671


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- GRÁFICO 2: las dos tasas del embudo por canal (barras agrupadas).
-- Aporta valor porque muestra de un vistazo dónde se estrecha el embudo:
-- la barra chica (impresión->clic, ~2%) es la etapa problema; la otra (~6-10%) cierra bien.
-- Truco para 2 medidas en un mismo eje Y: se "apilan" en una columna de nombre y otra de valor (UNPIVOT).
SELECT canal, 'impresion -> clic' AS tasa, tasa_impresion_a_clic_pct AS valor_pct
FROM dev.gold_ads.q5_embudo_por_canal
UNION ALL
SELECT canal, 'clic -> conversion' AS tasa, tasa_clic_a_conversion_pct AS valor_pct
FROM dev.gold_ads.q5_embudo_por_canal
ORDER BY canal, tasa;

canal,tasa,valor_pct
Display,clic -> conversion,9.744
Display,impresion -> clic,1.941
Search,clic -> conversion,5.976
Search,impresion -> clic,2.118
Social,clic -> conversion,7.885
Social,impresion -> clic,1.948
Video,clic -> conversion,8.662
Video,impresion -> clic,1.959


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- GRÁFICO 3: volumen diario de impresiones en la ventana de 30 días de octubre (línea).
-- Aporta valor porque agrega la dimensión que las tablas no muestran: el TIEMPO.
-- Se limita a octubre porque los 2 días de noviembre están aislados del rango
-- (dejarlos rompería la línea con un salto de 8 días).
SELECT to_date(event_ts) AS dia, COUNT(*) AS impresiones
FROM dev.silver_ads.impressions
WHERE to_date(event_ts) BETWEEN '2026-10-01' AND '2026-10-30'
GROUP BY 1
ORDER BY 1;

dia,impresiones
2026-10-01,5000
2026-10-02,5000
2026-10-03,5000
2026-10-04,5000
2026-10-05,5000
2026-10-06,5000
2026-10-07,5000
2026-10-08,5000
2026-10-09,5000
2026-10-10,5000


Databricks visualization. Run in Databricks to view.